# LINEAR REGRESSION
# Thông tin sinh viên
MSSV: 24127095

Họ và tên: Vũ Duy Nhất
# Thông tin giải thuật

- Phương pháp OLS được Adrien-Marie Legendre công bố vào năm 1805 và Carl Friedrich Gauss phát triển độc lập từ năm 1795 nhằm giải quyết các bài toán thiên văn học, tiêu biểu là việc xác định quỹ đạo tiểu hành tinh. Gauss đã kết hợp OLS với phân phối xác suất chuẩn và chứng minh Định lý Gauss-Markov, khẳng định đây là ước lượng tuyến tính không chệch tốt nhất (BLUE). Từ một công cụ tính toán thiên văn ban đầu, OLS đã phát triển thành nền tảng cốt lõi của thống kê, kinh tế lượng và khoa học dữ liệu hiện đại.

- OLS xác định đường hồi quy tối ưu bằng cách tối thiểu hóa tổng bình phương các khoảng cách thẳng đứng (phần dư) giữa giá trị thực tế $y$ và giá trị dự đoán $\hat{y}$. Việc bình phương phần dư giúp triệt tiêu dấu âm và gia tăng mức phạt đối với các điểm dữ liệu lệch xa. Bằng cách lấy đạo hàm hàm mất mát theo tập hệ số $\beta$ và cho bằng $0$, thuật toán giải hệ phương trình chuẩn để đưa ra công thức dạng đóng $\hat{\beta} = (X^T X)^{-1} X^T y$, cho phép tính toán trực tiếp bộ thông số tối ưu từ dữ liệu đầu vào.

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
import math

df = pd.read_csv("wine.csv")

def dataParitionInQuestionA():
    x = df.drop(columns='"quality"')
    y = df['"quality"']

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    feature_names = x_train.columns.tolist()
    target_names = y_train.name

    x_train = x_train.values.tolist()
    x_test = x_test.values.tolist()

    y_train = y_train.values.tolist()
    y_test = y_test.values.tolist()

    return x_train, x_test, y_train, y_test, feature_names, target_names

def dataParitionInQuestionB():
    feature_cols = [col for col in df.columns if col != '"quality"']                                                                                                                                                
    y = df['"quality"']                                                                                                                                                                                             
                                                                                                                                                                                                                    
    result = {}                                                                                                                                                                                                     
    for feature in feature_cols:                                                                                                                                                                                    
        x = df[[feature]]   # single-column DataFrame → 2D                                                                                                                                                          
                                                                                                                                                                                                                    
        x_train, x_test, y_train, y_test = train_test_split(                                                                                                                                                        
            x, y, test_size=0.2, random_state=42                                                                                                                                                                    
        )                                                                                                                                                                                                           
                                                                                                                                                                                                                    
        result[feature] = {                                                                                                                                                                                         
            "x_train": x_train.values.tolist(),                                                                                                                                                                     
            "x_test":  x_test.values.tolist(),                                                                                                                                                                      
            "y_train": y_train.values.tolist(),                                                                                                                                                                     
            "y_test":  y_test.values.tolist(),                                                                                                                                                                      
        }                                                                                                                                                                                                           
                                                                                                                                                                                                                    
    return result      


In [22]:
def relativeCompare(num1, num2, eps = 1e-9):
    return abs(num1 - num2) <= eps

def unitMatrix(n):
    return [[1 if i == j else 0 for j in range(n)] for i in range(n)]

def connectMatrixWithMatrix(M1, M2):
    M = M1
    n = len(M1)
    if n != len(M2):
        raise ValueError("Ma trận phải có cùng số hàng để kết nối.")
    for i in range(n):
        M[i].extend(M2[i])
    return M

def getColumnVector(M,i):
    res = []
    for row in M:
        res.append(row[i])
    return res

In [23]:
def swap_row(M, i, j):
    M[i],M[j] = M[j],M[i]

def multiply(M, i, alpha):
    n = len(M[i])
    for x in range (n):
        M[i][x] = M[i][x]*alpha

def eliminate_row(M, i, j, k):
    n = len(M[i])
    for x in range (n):
        val = M[j][x] - k*M[i][x]
        M[j][x] = 0 if relativeCompare(val,0) else val

In [24]:
def multiplyMatrixWithVector(M, v):
    n = len(M)
    m = len(v)
    if (len(M[0]) != m):
        raise ValueError("Số cột trong M phải bằng số hàng trong v.")
    result = []
    for i in range(n):
        sum_product = 0
        for j in range(m):
            sum_product += M[i][j] * v[j]
        if relativeCompare(sum_product, 0):
            sum_product = 0
        result.append(sum_product)
    return result

def multiplyMatrixWithMatrix(M1,M2):
    n = len(M1)
    m = len(M2[0])
    p = len(M2)
    if len(M1[0]) != p:
        raise ValueError("Số cột trong M1 phải bằng số hàng trong M2.")
    result = []
    for i in range(n):
        row_result = []
        for j in range(m):
            sum_product = 0
            for k in range(p):
                sum_product += M1[i][k] * M2[k][j]
            if relativeCompare(sum_product, 0):
                sum_product = 0
            row_result.append(sum_product)
        result.append(row_result)
    return result

def transposeMatrix(M):
    res = []
    cols = len(M[0])
    for i in range(cols):
        res.append(getColumnVector(M,i))
    return res

def inverseMatrix(A):
    M = connectMatrixWithMatrix(A, unitMatrix(len(A)))
    num_rows = len(M)
    
    n = num_rows 
    
    pivot_row = 0
    pivot_col = 0
    
    while pivot_row < n and pivot_col < n:
        
        nonzero_row = pivot_row
        while nonzero_row < num_rows and relativeCompare(M[nonzero_row][pivot_col], 0):
            nonzero_row += 1

        if nonzero_row == num_rows:
            print(f"Lỗi: Ma trận không có ma trận nghịch đảo!")
            return None
            
        if nonzero_row != pivot_row:
            swap_row(M, pivot_row, nonzero_row)
        
        pivot_val = M[pivot_row][pivot_col]
        if not relativeCompare(pivot_val, 1):
            multiply(M, pivot_row, 1 / pivot_val)
        
        for x in range(0, num_rows):
            if x == pivot_row:
                continue
            if not relativeCompare(M[x][pivot_col], 0):
                eliminate_row(M, pivot_row, x, M[x][pivot_col])
        
        pivot_row += 1
        pivot_col += 1

    A_inv = []
    for r in range(n):
        A_inv.append(M[r][n:])
        
    return A_inv

In [25]:
def modelBuildUp(beta_coefficients, feature_names=None):
    """Hàm biểu diễn mô hình hồi quy tuyến tính dưới dạng phương trình toán học."""
    if feature_names is None:
        feature_names = [
            "fixed acidity",
            "volatile acidity",
            "citric acid",
            "residual sugar",
            "chlorides",
            "free sulfur dioxide",
            "total sulfur dioxide",
            "density",
            "pH",
            "sulphates",
            "alcohol",
        ]

    b = beta_coefficients[0]
    equation = f"Quality = {b:.4f}"

    for i, name in enumerate(feature_names):
        coef = beta_coefficients[i + 1]
        sign = " + " if coef >= 0 else " - "
        equation += f"{sign}{abs(coef):.4f} * ({name})"

    return equation

def insert_intercept(M):
    X = []
    for i in range (len(M)):
            row_data = [1.0] + [float(M[i][col]) for col in range(len(M[0]))]
            X.append(row_data)
    return X

def predict(x_test, theta):

    X_test = insert_intercept(x_test)
    predicted_quality = multiplyMatrixWithVector(X_test, theta)

    return predicted_quality

def metric(y_test, y_pred):
    n = len(y_test)
    mse = sum((y_test[i] - y_pred[i]) ** 2 for i in range(n)) / n
    rmse = mse ** 0.5
    return rmse

def coef_OLS(x_train, y_train):
    X_train = insert_intercept(x_train)
    a = inverseMatrix(multiplyMatrixWithMatrix(transposeMatrix(X_train), X_train))
    b = multiplyMatrixWithVector(transposeMatrix(X_train), y_train)
    theta = multiplyMatrixWithVector(a, b)
    return theta

In [26]:
def k_fold(x_train, y_train, k):
    n = len(x_train)                                                                                                                                                                                                
    fold_size = n // k                                                                                                                                                           
    remainder = n % k                                                                                                                                       
                                                                                                                                                                                                                    
    folds = []                                                                                                                                                                                                      
    start = 0                                                                                                                                                                                                       
    for i in range(k):                                                                                                                                                                                              
        end = start + fold_size + (1 if i < remainder else 0)                                                                                                                                                       
        folds.append((x_train[start:end], y_train[start:end]))                                                                                                                                                      
        start = end                                                                                                                                                                                                 
                                                                                                                                                                                                                    
    return folds    

def cross_validation(x_train, y_train, k):
    folds = k_fold(x_train, y_train, k)                                                                                                                                                                             
    rmse_list = []                                                                                                                                                                                                  
                                                                                                                                                                                                                    
    for i in range(k):                                                                                                                                                                                                                                                                                                                                                                     
        x_valid, y_valid = folds[i]                                                                                                                                                                                     
                                                                                                                                                                                                                    
                                                                                                                                                   
        x_cv_train = []                                                                                                                                                                                             
        y_cv_train = []                                                                                                                                                                                             
        for j in range(k):                                                                                                                                                                                          
            if j != i:                                                                                                                                                                                              
                x_cv_train += folds[j][0]                                                                                                                                                                           
                y_cv_train += folds[j][1]                                                                                                                                                                           
                                                                                                                                                                                                                    
                                                                                                                                                                                 
        theta   = coef_OLS(x_cv_train, y_cv_train)                                                                                                                                                                  
        y_pred  = predict(x_valid, theta)                                                                                                                                                                             
        rmse    = metric(y_valid, y_pred)                                                                                                                                                                             
        rmse_list.append(rmse)                                                                                                                                                                                      
                                                                                                                                                                                                                                                                                                                                                                             
                                                                                                                                                                                                                    
    avg_rmse = sum(rmse_list) / k                                                                                                                                                                                                                                                                                                                                      
    return avg_rmse   

In [27]:
def transform_feature(col, mode):                                                                                                                                                                                                                                                                                                                                                                                           
    result = []                                                                                                                                                                                                     
    for v in col:                                                                                                                                                                                                   
        if mode == 'identity':                                                                                                                                                                                      
            result.append(v)                                                                                                                                                                                        
        elif mode == 'log':                                                                                                                                                                                         
            result.append(math.log(v) if v > 0 else float('nan'))                                                                                                                                                   
        elif mode == 'sqrt':                                                                                                                                                                                        
            result.append(math.sqrt(v) if v >= 0 else float('nan'))                                                                                                                                                 
        elif mode == 'square':                                                                                                                                                                                      
            result.append(v ** 2)                                                                                                                                                                                   
        elif mode == 'reciprocal':                                                                                                                                                                                  
            result.append(1 / v if v != 0 else float('nan'))                                                                                                                                                        
        elif mode == 'log1p':                                                                                                                                                                                       
            result.append(math.log1p(v) if v > -1 else float('nan'))                                                                                                                                                
        else:                                                                                                                                                                                                       
            raise ValueError(f"Unknown mode: {mode}")                                                                                                                                                               
    return result  

def preprocess_data(x_data, transform_map):
    n_rows = len(x_data)
    n_cols = len(x_data[0]) if n_rows > 0 else 0
    transformed_cols = []
    for i in range(n_cols):
        col = [x_data[j][i] for j in range(n_rows)]
        mode = transform_map[i] if i < len(transform_map) else 'identity'
        transformed_cols.append(transform_feature(col, mode))
    return transposeMatrix(transformed_cols)

def modelBuildUpWithTransforms(theta, transform_map, feature_names):                                                                                                                                                
    b = theta[0]                                                                                                                                                                                                    
    equation = f"Quality = {b:.4f}"                                                                                                                                                                                 
                                                                                                                                                                                                                    
    transform_symbol = {                                                                                                                                                                                            
        'identity'  : '{name}',                                                                                                                                                                                     
        'log'       : 'ln({name})',                                                                                                                                                                                 
        'sqrt'      : '√({name})',                                                                                                                                                                                  
        'square'    : '({name})²',                                                                                                                                                                                  
        'reciprocal': '1/({name})',                                                                                                                                                                                 
        'log1p'     : 'ln(1 + {name})',                                                                                                                                                                             
    }                                                                                                                                                                                                               
                                                                                                                                                                                                                    
    for i, (name, mode) in enumerate(zip(feature_names, transform_map)):                                                                                                                                            
        coef = theta[i + 1]                                                                                                                                                                                         
        sign = " + " if coef >= 0 else " - "                                                                                                                                                                        
        term = transform_symbol[mode].format(name=name)                                                                                                                                                             
        equation += f"{sign}{abs(coef):.4f} * {term}"                                                                                                                                                               
                                                                                                                                                                                                                    
    return equation

In [28]:
# Câu A
x_train, x_test, y_train, y_test, feature_names, target_names = dataParitionInQuestionA()
theta = coef_OLS(x_train, y_train)
print(modelBuildUp(theta,feature_names))
y_predict = predict(x_test, theta)
rmse = metric(y_test, y_predict)
print("RMSE:", rmse)

Quality = 46.0179 + 0.0491 * ("fixed acidity") - 1.0472 * ("volatile acidity") - 0.2689 * ("citric acid") + 0.0505 * ("residual sugar") - 1.4910 * ("chlorides") + 0.0047 * ("free sulfur dioxide") - 0.0043 * ("total sulfur dioxide") - 42.7098 * ("density") - 0.2156 * ("pH") + 0.7476 * ("sulphates") + 0.2667 * ("alcohol")
RMSE: 0.6319990258040524


In [29]:
# Câu B
processed_data = dataParitionInQuestionB()
avg_rmse_dict = {}
for feature, data in processed_data.items():
    x_train = data["x_train"]
    y_train = data["y_train"]
    avg_rmse = cross_validation(x_train, y_train, k=5)
    avg_rmse_dict[feature] = avg_rmse

for feature, avg_rmse in avg_rmse_dict.items():
    print(f"Feature: {feature}, Average RMSE: {avg_rmse:.4f}")

best_feature = min(avg_rmse_dict, key=avg_rmse_dict.get)
print(f"\n--> Best feature for predicting wine quality: {best_feature} with Average RMSE: {avg_rmse_dict[best_feature]:.4f}")

theta_best = coef_OLS(processed_data[best_feature]["x_train"], processed_data[best_feature]["y_train"])
print()
print(modelBuildUp(theta_best, [best_feature]))
y_predict_best = predict(processed_data[best_feature]["x_test"], theta_best)
rmse_best = metric(processed_data[best_feature]["y_test"], y_predict_best)
print(f"RMSE for best feature ({best_feature}):", rmse_best)

Feature: "fixed acidity", Average RMSE: 0.8041
Feature: "volatile acidity", Average RMSE: 0.7506
Feature: "citric acid", Average RMSE: 0.7914
Feature: "residual sugar", Average RMSE: 0.8122
Feature: "chlorides", Average RMSE: 0.8087
Feature: "free sulfur dioxide", Average RMSE: 0.8125
Feature: "total sulfur dioxide", Average RMSE: 0.7929
Feature: "density", Average RMSE: 0.7996
Feature: "pH", Average RMSE: 0.8107
Feature: "sulphates", Average RMSE: 0.7957
Feature: "alcohol", Average RMSE: 0.6999

--> Best feature for predicting wine quality: "alcohol" with Average RMSE: 0.6999

Quality = 1.7253 + 0.3785 * ("alcohol")
RMSE for best feature ("alcohol"): 0.6945930064921848


In [30]:
# Câu C
x_train, x_test, y_train, y_test, feature_names, target_names = dataParitionInQuestionA()
TRANSFORMS = ['identity', 'log', 'sqrt', 'square', 'reciprocal', 'log1p']

n = len(feature_names)
best_map = ['identity'] * n
for i in range(n):
    best_rmse = float('inf')
    best_transform = 'identity'
    for mode in TRANSFORMS:
        trial_map = best_map[:]
        trial_map[i] = mode
        x_tr = preprocess_data(x_train, trial_map)
        if any(math.isnan(val) for row in x_tr for val in row):
            continue
        avg_rmse = cross_validation(x_tr, y_train, k=5)
        if avg_rmse < best_rmse:
            best_rmse = avg_rmse
            best_transform = mode
    best_map[i] = best_transform
    print(f"Best transformation for feature '{feature_names[i]}' is {best_transform} with Average RMSE: {best_rmse:.4f}\n")

print()
x_tr_best = preprocess_data(x_train, best_map)
theta_best = coef_OLS(x_tr_best, y_train)
print(modelBuildUpWithTransforms(theta_best, best_map, feature_names))
x_te_best = preprocess_data(x_test, best_map)
y_predict_best = predict(x_te_best, theta_best)
rmse_best = metric(y_test, y_predict_best)
print(f"RMSE after applying best transformations: {rmse_best}")



Best transformation for feature '"fixed acidity"' is reciprocal with Average RMSE: 0.6486

Best transformation for feature '"volatile acidity"' is identity with Average RMSE: 0.6486

Best transformation for feature '"citric acid"' is sqrt with Average RMSE: 0.6477

Best transformation for feature '"residual sugar"' is log with Average RMSE: 0.6473

Best transformation for feature '"chlorides"' is reciprocal with Average RMSE: 0.6463

Best transformation for feature '"free sulfur dioxide"' is log with Average RMSE: 0.6455

Best transformation for feature '"total sulfur dioxide"' is identity with Average RMSE: 0.6455

Best transformation for feature '"density"' is square with Average RMSE: 0.6455

Best transformation for feature '"pH"' is square with Average RMSE: 0.6455

Best transformation for feature '"sulphates"' is reciprocal with Average RMSE: 0.6368

Best transformation for feature '"alcohol"' is log1p with Average RMSE: 0.6367


Quality = 41.0669 - 7.0227 * 1/("fixed acidity") - 

# Ý tưởng thực hiện và mô tả các hàm
## 1. Tiện ích
- `pandas`: Thư viện này hỗ trợ việc đọc dữ liệu từ file csv vào dataframe $df$.

- `train_test_split() từ scikit-learn`: Hàm này hỗ trợ phân hoạch dữ liệu đầu vào thành tập huấn luyện (training set) và tập kiểm thử (test set), phục vụ việc huấn luyện và đánh giá mô hình.

- `math`: Thư viện này cung cấp các hàm tính $ln(x), \sqrt(x), ln(1+x)$, được dùng trong việc tự xây dựng mô hình ở yêu cầu C.

- `relativeCompare(num1, num2, eps = 1e-9)`: Hàm này dùng để so sánh bằng giữa hai số $a$, $b$ bằng cách so sánh hiệu của chúng xem có bé hơn hoặc bằng sai số cực nhỏ $eps$ cho trước hay không. Thay vì dùng so sánh `==` truyền thống, có thể dẫn đến kết quả sai do vấn đề làm tròn số hoặc biểu diễn dấu chấm động, việc so sánh sử dùng hàm này sẽ đảm bảo kết quả đúng khi thao tác với số thực.

- `unitMatrix(n)`: Hàm này trả về ma trận đơn vị $I$ tương ứng với kích $n$ truyền vào.

- `connectMatrixWithMatrix(M1, M2)`: Hàm này dùng để ghép hai ma trận $M1$ và $M2$ đầu vào để tạo ra ma trận mở rộng dạng $(M1|M2)$.

- `getColumnVector(M,i)`: Hàm này trả về vector cột $i$ của ma trận $M$

## 2. Các phép biến đổi sơ cấp trên dòng
Các phép biến đổi sơ cấp trong đồ án này được dùng bởi **giải thuật Gauss-Jordan** để tìm ma trận nghịch đảo.

- `swap_row(M, i, j)`: Hàm này nhận 3 tham số, ma trận là $M$, chỉ số hàng i là $i$ và chỉ số hàng j là $j$, và thực hiện hoán vị dòng i và dòng j trong ma trận M.

- `multiply(M, i, alpha)`:  Hàm này nhận 3 tham số, ma trận là $M$, chỉ số hàng i là $i$ và một số thực $alpha$, và thực hiện nhân số thực alpha vào từng phần tử trong dòng i của ma trận M.

- `eliminate_row(M, i, j, k)`: Hàm này 4 tham số, ma trận là $M$, chỉ số hàng i là $i$, chỉ số hàng j là $j$ và hệ số $k$, và thực hiện tính toán giá trị mới cho $dòng_j$ theo công thức: $dòng_j = dòng_j - k\times dòng_i$.

## 3. Các phép tính trên ma trận
- `multiplyMatrixWithVector(M, v)`: Hàm này nhận vào ma trận $M$ và vector cột $v$, thực hiện phép nhân ma trận với vector và trả về vector kết quả $result$, với mỗi phần tử được tính theo công thức $result_i = \sum_{j=1}^{m} M_{ij} \cdot v_j$. Trước khi thực hiện phép nhân, hàm kiểm tra số cột của $M$ phải bằng số chiều của $v$; ngoài ra, các giá trị kết quả xấp xỉ 0 (theo relativeCompare) được làm tròn về 0 để tránh sai số dấu chấm động tích lũy.

- `multiplyMatrixWithMatrix(M1,M2)`: Hàm này nhận ma trận $M1$, ma trận $M2$ và thực hiện phép nhân ma trận $M1 \times M2 $và trả về ma trận kết quả $result$, với mỗi phần tử được tính theo công thức $result_{ij} = \sum_{k=1}^{p} M1_{ik} \cdot M2_{kj}$. Trước khi thực hiện phép nhân ma trận, phải kiểm tra $M1$ và $M2$ để đảm bảo số dòng của ma trận $M1$ bằng số dòng của ma trận $M2$ $(len(M1[0]) == len(M2))$.

- `transposeMatrix(M)`: Hàm này dùng để tìm ma trận nghịch đảo của ma trận $M$, các cột của ma trận hiện tại trở thành các dòng trong ma trận chuyển vị.

- `inverseMatrix(A)`: Hàm nhận vào ma trận $A$ (ở đây là ma trận $P$) và thực hiện giải thuật Gauss-Jordan trên ma trận mở rộng $M = (A | I)$ để tìm ma trận nghịch đảo $A^{-1}$.

## 4. Phân hoạch dữ liệu
- `dataParitionInQuestionA()`: Hàm này thực hiện phân hoạch dữ liệu phục vụ câu A, sử dụng toàn bộ 11 đặc trưng đầu vào để dự đoán biến mục tiêu $quality$. Từ dataframe $df$ ban đầu, hàm tách ra tập đặc trưng $x$ (loại bỏ cột "quality") và tập nhãn $y$ (cột "quality"), sau đó dùng `train_test_split` để chia thành tập huấn luyện (80%) và tập kiểm thử (20%) với `random_state=42` nhằm đảm bảo tính tái lập của kết quả. Cuối cùng, các DataFrame/Series được chuyển về dạng danh sách lồng nhau (list of lists) để tương thích với các hàm ma trận tự cài đặt, đồng thời hàm trả về thêm danh sách tên đặc trưng (feature_names) và tên biến mục tiêu (target_names).

- `dataParitionInQuestionB():`: Hàm này thực hiện phân hoạch dữ liệu phục vụ câu B, với mục tiêu tìm ra đặc trưng đơn lẻ cho khả năng dự đoán tốt nhất. Với mỗi đặc trưng trong 11 đặc trưng, hàm tách riêng cột đặc trưng đó thành tập $x$ (dưới dạng DataFrame một cột để giữ định dạng hai chiều) và cột "quality" làm tập $y$, sau đó dùng `train_test_split` để chia tập huấn luyện/kiểm thử tương ứng. Kết quả trả về là một dictionary với khóa là tên từng đặc trưng và giá trị là dictionary con chứa 4 tập dữ liệu (x_train, x_test, y_train, y_test) dạng danh sách.

## 5. Bài làm
### 5.1. Câu A: Xây dựng mô hình bằng việc sử dụng 11 đặc trưng để bài cung cấp
#### 5.1.1. Các hàm liên quan
- `modelBuildUp(beta_coefficients, feature_names=None)`: Hàm này nhận vào vector hệ số hồi quy $\hat\beta$ tìm được từ OLS và danh sách tên các đặc trưng tương ứng, sau đó xây dựng và trả về chuỗi biểu diễn tường minh của phương trình hồi quy tuyến tính dạng $Quality = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$, giúp trực quan hóa mô hình thu được sau khi huấn luyện.

- `insert_intercept(M)`: Hàm này chèn thêm một cột có giá trị $1$ vào đầu mỗi dòng của ma trận đặc trưng $M$, nhằm chuyển ma trận $X$ kích thước $(n \times p)$ thành ma trận mở rộng $(n \times (p+1))$, phục vụ việc tính toán hệ số chặn $\beta_0$ khi giải hệ phương trình chuẩn theo dạng ma trận.

- `predict(x_test, theta)`: Hàm này thực hiện dự đoán giá trị đầu ra dựa trên tập dữ liệu $x\_test$ và vector hệ số $\theta$ đã học được. Đầu tiên, hàm chèn cột hệ số chặn vào $x\_test$ bằng `insert_intercept` để thu được ma trận $X_{test}$, sau đó nhân ma trận $X_{test}$ với vector $\theta$ (dùng `multiplyMatrixWithVector`) để tính ra vector giá trị dự đoán $\hat y$.

- `metric(y_test, y_pred)`: Hàm này tính toán độ đo Root Mean Squared Error (RMSE) giữa giá trị thực tế $y\_test$ và giá trị dự đoán $y\_pred$, theo công thức $RMSE = \sqrt{\dfrac{1}{n}\sum_{i=1}^n (y_i - \hat y_i)^2}$, dùng làm tiêu chí đánh giá độ chính xác của mô hình hồi quy.

-  `coef_OLS(x_train, y_train)`: Hàm này tính toán vector hệ số hồi quy tối ưu $\hat\beta$ theo công thức nghiệm đóng của OLS $\hat\beta = (X^T X)^{-1} X^T y$. Đầu tiên, ma trận đặc trưng $x\_train$ được chèn thêm cột hệ số chặn thông qua `insert_intercept` để tạo ma trận $X$. Sau đó, hàm tính ma trận nghịch đảo $(X^TX)^{-1}$ bằng `inverseMatrix`, tính vector $X^Ty$ bằng `multiplyMatrixWithVector`, và cuối cùng nhân hai kết quả trên để thu được vector hệ số $\theta$.

#### 5.1.2. Ý tưởng thực hiện
- Sử dụng toàn bộ 11 đặc trưng đầu vào để dự đoán chất lượng rượu vang (quality).

  1. Phân hoạch dữ liệu thành tập huấn luyện và tập kiểm thử theo tỉ lệ 8:2 bằng hàm `dataParitionInQuestionA()`.

  2. Đưa tập huấn luyện vào hàm `coef_OLS` để tính vector hệ số $\hat\beta$ theo công thức nghiệm đóng của OLS, dựa trên các phép toán ma trận tự cài đặt (nhân ma trận, chuyển vị, nghịch đảo bằng giải thuật Gauss-Jordan).

  3. Biểu diễn tường minh phương trình hồi quy từ hệ số $\hat\beta$ tìm được bằng hàm `modelBuildUp`.

  4. Đánh giá mô hình trên tập kiểm thử: dự đoán giá trị quality bằng hàm `predict`, sau đó tính RMSE giữa giá trị dự đoán và giá trị thực tế bằng hàm `metric`.

### 5.2. Câu B: Xây dựng mô hình bằng việc sử dụng đặc trưng cho kết quả tốt nhất, dùng phương pháp Cross Validation
#### 5.2.1. Các hàm liên quan
- `k_fold(x_train, y_train, k)`: Hàm này chia tập dữ liệu huấn luyện $(x\_train, y\_train)$ thành $k$ phần (fold) có kích thước gần bằng nhau. Cụ thể, hàm tính kích thước cơ bản của mỗi fold bằng phép chia nguyên ($n // k$) và phần dư ($n \mod k$), sau đó phân phối phần dư đều cho các fold đầu tiên để đảm bảo chênh lệch kích thước giữa các fold không quá 1 phần tử. Kết quả trả về là danh sách các cặp $(x\_fold, y\_fold)$ tương ứng với từng fold.

- `cross_validation(x_train, y_train, k)`: Hàm này thực hiện đánh giá mô hình bằng phương pháp k-fold cross validation. Với mỗi fold $i$ trong $k$ fold (lấy từ `k_fold`), fold $i$ được dùng làm tập validation, các fold còn lại được gộp lại làm tập huấn luyện tạm thời. Mô hình được huấn luyện trên tập huấn luyện tạm thời bằng `coef_OLS`, sau đó dùng để dự đoán và tính RMSE trên tập validation (hàm `predict` và `metric`). Cuối cùng, hàm trả về giá trị RMSE trung bình qua $k$ lần lặp, dùng làm tiêu chí đánh giá độ tốt của từng đặc trưng trong câu B.

#### 5.2.2. Ý tưởng thực hiện
- Mục tiêu: tìm ra đặc trưng đơn lẻ có khả năng dự đoán quality tốt nhất trong số 11 đặc trưng ban đầu.

  1. Với mỗi đặc trưng, phân hoạch dữ liệu riêng bằng `dataParitionInQuestionB()`.
  
  2. Đánh giá tập huấn luyện tương ứng của từng đặc trưng bằng phương pháp 5-fold cross validation (hàm `cross_validation`) để tính RMSE trung bình.

  3. Chọn đặc trưng cho RMSE trung bình nhỏ nhất làm đặc trưng tốt nhất (best_feature).
  
  4. Huấn luyện lại mô hình OLS trên toàn bộ tập huấn luyện của đặc trưng tốt nhất này, sau đó đánh giá trên tập kiểm thử tương ứng để thu được RMSE cuối cùng.

### 5.3. Câu C: Xây dựng mô hình riêng cho kết quả tốt nhất
#### 5.3.1. Các hàm liên quan
- `transform_feature(col, mode)`: Hàm này thực hiện biến đổi (transform) một cột dữ liệu $col$ theo phép biến đổi được chỉ định bởi tham số $mode$, bao gồm các phép: giữ nguyên (identity), logarit tự nhiên (log), căn bậc hai (sqrt), bình phương (square), nghịch đảo (reciprocal) và $ln(1+x)$ (log1p). Với các phép biến đổi có miền xác định giới hạn (log, sqrt, reciprocal, log1p), nếu giá trị đầu vào không hợp lệ (ví dụ $v \le 0$ với log), hàm trả về giá trị `nan` để đánh dấu, tránh gây lỗi khi tính toán.

- `preprocess_data(x_data, transform_map)`: Hàm này áp dụng phép biến đổi tương ứng cho từng cột của ma trận đặc trưng $x\_data$, dựa trên danh sách các phép biến đổi $transform\_map$ (mỗi phần tử trong $transform\_map$ tương ứng với một cột đặc trưng). Với mỗi cột, hàm trích xuất giá trị của cột đó, gọi `transform_feature` để biến đổi, sau đó dùng `transposeMatrix` để ghép các cột đã biến đổi trở lại thành ma trận đặc trưng mới.

- `modelBuildUpWithTransforms(theta, transform_map, feature_names)`: Tương tự hàm `modelBuildUp`, hàm này xây dựng chuỗi biểu diễn phương trình hồi quy, nhưng có thêm ký hiệu toán học tương ứng với phép biến đổi đã áp dụng cho từng đặc trưng (ví dụ $ln(name)$, $\sqrt{name}$, $(name)^2$, $1/(name)$, $ln(1+name)$) dựa trên $transform\_map$, giúp phương trình phản ánh đúng dạng mô hình sau khi biến đổi đặc trưng.

#### 5.3.2. Ý tưởng thực hiện
- Mục tiêu: xây dựng mô hình hồi quy riêng cho toàn bộ 11 đặc trưng bằng cách thử áp dụng các phép biến đổi phi tuyến khác nhau lên từng đặc trưng, nhằm cải thiện khả năng dự đoán so với mô hình tuyến tính thông thường ở câu A.

  1. Với mỗi đặc trưng, lần lượt thử từng phép biến đổi trong tập TRANSFORMS (identity, log, sqrt, square, reciprocal, log1p) áp dụng riêng cho đặc trưng đó, trong khi các đặc trưng còn lại vẫn giữ nguyên phép biến đổi tốt nhất đã tìm được ở các bước trước (chiến lược tham lam - greedy, tối ưu từng đặc trưng một).
  
  2. Với mỗi lựa chọn, biến đổi dữ liệu bằng `preprocess_data` rồi đánh giá bằng 5-fold cross validation; các trường hợp sinh ra giá trị `nan` (do vi phạm miền xác định) sẽ bị bỏ qua.

  3. Giữ lại phép biến đổi cho RMSE trung bình nhỏ nhất cho đặc trưng đang xét.

  4. Sau khi xác định phép biến đổi tối ưu cho toàn bộ 11 đặc trưng (best_map), biến đổi lại tập huấn luyện và tập kiểm thử theo best_map.

  5. Huấn luyện mô hình OLS trên tập huấn luyện đã biến đổi (`coef_OLS`), và đánh giá bằng RMSE trên tập kiểm thử đã biến đổi tương ứng (`predict`, `metric`).

## 6. So sánh kết quả 3 mô hình từ bài làm
Ba mô hình được xây dựng trong bài tương ứng với ba chiến lược khác nhau:

- **Mô hình A** (11 đặc trưng gốc, không biến đổi): sử dụng toàn bộ thông tin đầu vào nên có nhiều khả năng nắm bắt được mối quan hệ giữa các đặc trưng và biến mục tiêu hơn so với mô hình chỉ dùng 1 đặc trưng, tuy nhiên đây vẫn là mô hình tuyến tính thuần túy, không có biến đổi phi tuyến.

- **Mô hình B** (1 đặc trưng tốt nhất, chọn qua cross validation): là mô hình đơn giản nhất, chỉ dùng duy nhất đặc trưng có tương quan mạnh nhất với quality. Do bỏ qua thông tin từ các đặc trưng còn lại, RMSE của mô hình này thường cao hơn (kém hơn) so với mô hình A.

- **Mô hình C** (11 đặc trưng, có biến đổi phi tuyến tối ưu theo từng đặc trưng): kế thừa toàn bộ thông tin của mô hình A, đồng thời áp dụng các phép biến đổi phi tuyến (log, sqrt, square, reciprocal, log1p) để tuyến tính hóa tốt hơn mối quan hệ giữa từng đặc trưng và quality, nên thường cho RMSE nhỏ nhất (tốt nhất) trong ba mô hình.

| Mô hình | Số đặc trưng sử dụng | Có biến đổi phi tuyến | RMSE (tập kiểm thử) |
|---|---|---|---|
| A - 11 đặc trưng gốc | 11 | Không | 0.6320 |
| B - đặc trưng tốt nhất (`alcohol`) | 1 | Không | 0.6946 |
| C - 11 đặc trưng đã biến đổi | 11 | Có | 0.6256 |

Kết quả thực nghiệm trên tập dữ liệu `wine.csv` cho thấy đúng như dự đoán: **mô hình C** (11 đặc trưng, có biến đổi phi tuyến tối ưu theo từng đặc trưng) cho RMSE thấp nhất (0.6256), tức là dự đoán chính xác nhất; **mô hình A** (11 đặc trưng gốc, không biến đổi) đứng thứ hai với RMSE = 0.6320, nhỉnh hơn không đáng kể so với mô hình C; còn **mô hình B** (chỉ dùng 1 đặc trưng `alcohol` - đặc trưng có tương quan mạnh nhất với quality) cho RMSE cao nhất (0.6946) do đã bỏ qua thông tin từ 10 đặc trưng còn lại. Điều này cho thấy việc bổ sung các phép biến đổi phi tuyến phù hợp (như $ln$, $\sqrt{}$, nghịch đảo,...) tuy chỉ giúp cải thiện RMSE ở mức khiêm tốn so với mô hình tuyến tính đầy đủ đặc trưng, nhưng vẫn mang lại hiệu quả tốt hơn so với việc chỉ dùng một đặc trưng đơn lẻ.